<h3>한국어 임베딩과 캐시</h3>
<p><strong>개념</strong> 캐시는 같은 텍스트와 같은 모델 설정으로 만든 벡터를 다시 사용합니다. 키에는 텍스트, 모델, 차원처럼 벡터 공간을 결정하는 값을 포함합니다.</p>

In [1]:
# %pip은 환경 변수와 LangChain OpenAI 임베딩 패키지를 현재 커널에 설치합니다.
# 버전을 고정해 OpenAIEmbeddings의 사용법을 학생 환경과 맞춥니다.
%pip install -q "python-dotenv==1.2.3" "langchain-openai==1.6.0"


Note: you may need to restart the kernel to use updated packages.


In [2]:
# .env의 API 키를 읽고 256차원 임베딩 객체와 로컬 캐시 폴더를 준비합니다.
# mkdir의 parents와 exist_ok는 상위 폴더 생성과 기존 폴더 재사용을 허용합니다.
from dotenv import find_dotenv, load_dotenv
from langchain_openai import OpenAIEmbeddings
from pathlib import Path
import hashlib, json

load_dotenv(find_dotenv(usecwd=True))
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=256)
cache_dir = Path.cwd() / ".cache/ch10-notebook"
cache_dir.mkdir(parents=True, exist_ok=True)


<h3>실제 API 결과를 파일에 캐시하기</h3>
<p><strong>확인할 점</strong> 첫 실행은 <code>OpenAIEmbeddings.embed_documents()</code>를 호출하고, 같은 입력의 두 번째 요청은 저장된 벡터를 읽습니다.</p>

In [3]:
# 텍스트와 모델 설정을 JSON으로 직렬화한 뒤 SHA-256 해시를 캐시 파일명으로 사용합니다.
# embed_cached()는 파일이 있으면 벡터를 읽고, 없으면 API 반환값을 저장한 뒤 출처 문자열과 함께 반환합니다.
# unlink(missing_ok=True)는 이전 캐시를 지워 첫 호출이 API를 사용하게 합니다. 파일이 없어도 오류를 내지 않습니다.
# global api_calls는 함수 밖에 있는 호출 횟수 변수를 함수 안에서 갱신하도록 지정합니다.
texts = ["제주 배송은 이틀이 더 걸립니다.", "환불은 수령 후 7일 이내 신청합니다."]
setting = {"model": "text-embedding-3-small", "dimensions": 256}
key = hashlib.sha256(json.dumps([texts, setting], ensure_ascii=False).encode()).hexdigest()
path = cache_dir / f"{key}.json"; path.unlink(missing_ok=True)
api_calls = 0
def embed_cached() -> tuple[list[list[float]], str]:
    global api_calls
    if path.exists(): return json.loads(path.read_text()), "cache"
    value = embeddings.embed_documents(texts); api_calls += 1
    path.write_text(json.dumps(value)); return value, "openai"
vectors, first_source = embed_cached()
_, second_source = embed_cached()
print({"first": first_source, "second": second_source, "api_calls": api_calls,
       "vectors": len(vectors), "dimensions": len(vectors[0])})


{'first': 'openai', 'second': 'cache', 'api_calls': 1, 'vectors': 2, 'dimensions': 256}


<h3>일부러 실패해 보기</h3>
<p><strong>실패 사례</strong> 차원이 다른 벡터를 같은 인덱스에 넣으면 비교할 수 없습니다. 다음 셀은 혼합을 저장 전에 차단합니다.</p>

In [4]:
# 두 번째 벡터를 128차원으로 잘라 서로 다른 차원을 섞는 오류를 재현합니다.
# 집합은 중복 길이를 제거하므로 원소가 하나일 때만 모든 벡터 차원이 같습니다.
try:
    mixed = [vectors[0], vectors[1][:128]]
    assert len({len(vector) for vector in mixed}) == 1, "벡터 차원이 다릅니다"
except AssertionError as error:
    print(f"caught: {error}")


caught: 벡터 차원이 다릅니다


<p><strong>수정 방법</strong> 같은 모델과 차원으로 생성한 원본 벡터만 묶고, 저장 전에 차원을 다시 검증합니다.</p>

In [5]:
# 원본 벡터만 사용하고 각 벡터 길이의 집합이 256 하나인지 확인합니다.
# sorted()는 집합을 출력 순서가 고정된 리스트로 바꿉니다.
consistent = vectors
dimensions = {len(vector) for vector in consistent}
assert dimensions == {256}
print({"vectors": len(consistent), "dimensions": sorted(dimensions)})


{'vectors': 2, 'dimensions': [256]}
